In [1]:
import os
import copy
import gc
import ray
import json
import pickle as pkl
import pandas as pd
from tqdm import tqdm
import random
import numpy as np
from ray import tune
import datasets
import numpy as np
from sklearn import metrics
from datasets import load_metric

import torch
import scipy.sparse as sp
import numpy as np
import tempfile
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, SAGEConv, GCNConv
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.nn.aggr import PowerMeanAggregation, SoftmaxAggregation, SetTransformerAggregation, MLPAggregation, LSTMAggregation, SortAggregation, EquilibriumAggregation
from sklearn.utils.multiclass import unique_labels
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from utils import get_texts_for_ptype

from datasets import load_dataset
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler
import ray.train.huggingface.transformers
from ray.train import ScalingConfig
from ray.train.torch import TorchTrainer

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from filelock import FileLock
from torch.utils.data import random_split
from typing import Dict
import ray
from ray import train, tune
from ray.train import Checkpoint
from ray.tune.schedulers import ASHAScheduler

In [2]:
def load_data(data_dir="../data", ptype="is_rct"):
    
    x_tensor_path = os.path.join(data_dir, "x_tensor_sample.pkl")
    with open(x_tensor_path, "rb") as f:
        x_tensor = pkl.load(f)
    
    pmid2node_id_path = os.path.join(data_dir, "pmid2node_id_sample.json")
    with open(pmid2node_id_path, "r") as f:
        pmid2node_id = json.load(f)
    
    PMIDS = os.path.join(data_dir, "pmids.pkl") 
    with open(PMIDS, 'rb') as f:
        pmids = pkl.load(f)
    
    edge_index_path = os.path.join(data_dir, "edge_index_sample.pkl")
    with open(edge_index_path, "rb") as f:
        edge_index = pkl.load(f)
    
    num_nodes = len(pmid2node_id)
    train_mask = [False for _ in range(num_nodes)]
    val_mask = [False for _ in range(num_nodes)]
    test_mask = [False for _ in range(num_nodes)]
        
    train_pmids = pmids[ptype]['pos']['train'] + pmids[ptype]['neg']['train']
    val_pmids = pmids[ptype]['pos']['val'] + pmids[ptype]['neg']['val']
    test_pmids = pmids[ptype]['pos']['test'] + pmids[ptype]['neg']['test']
        
    for pmid in train_pmids:
        if str(pmid) in pmid2node_id:
            node_id = pmid2node_id[str(pmid)]
            train_mask[node_id] = True
        
    for pmid in val_pmids:
        if str(pmid) in pmid2node_id:
            node_id = pmid2node_id[str(pmid)]
            val_mask[node_id] = True
                
    for pmid in test_pmids:
        if str(pmid) in pmid2node_id:
            node_id = pmid2node_id[str(pmid)]
            test_mask[node_id] = True
        
    train_mask = torch.tensor(train_mask, dtype=torch.bool)
    val_mask = torch.tensor(val_mask, dtype=torch.bool)
    test_mask = torch.tensor(test_mask, dtype=torch.bool)
        
    y = [-1 for _ in range(num_nodes)]
    pos_pmids = pmids[ptype]['pos']['train'] + pmids[ptype]['pos']['val'] + pmids[ptype]['pos']['test']
    neg_pmids = pmids[ptype]['neg']['train'] + pmids[ptype]['neg']['val'] + pmids[ptype]['neg']['test']
        
    for pmid in pos_pmids:
        if str(pmid) in pmid2node_id:
            node_id = pmid2node_id[str(pmid)]
            y[node_id] = 1
        
    for pmid in neg_pmids:
        if str(pmid) in pmid2node_id:
            node_id = pmid2node_id[str(pmid)]
            y[node_id] = 0
        
    y = torch.tensor(y, dtype=torch.long)
    edge_index = torch.tensor(edge_index)
    data = Data(x=torch.tensor(x_tensor, dtype=torch.float32), y=y, edge_index=edge_index, train_mask=train_mask, val_mask=val_mask, test_mask=test_mask)
    del y
    del x_tensor
    del edge_index
    del train_mask
    del val_mask
    del test_mask
    
    return data


In [3]:
class GAT(torch.nn.Module):
    def __init__(self, hid, in_head, out_head, dropout, num_classes = 2):
        super(GAT, self).__init__()
        self.hid = hid
        self.in_head = in_head
        self.out_head = out_head
        
        
        self.conv1 = GATConv(5000, self.hid, heads=self.in_head, dropout=0.6)
        self.conv2 = GATConv(self.hid*self.in_head, num_classes, concat=False,
                             heads=self.out_head, dropout=0.6)
#         self.conv = GATConv(data.num_features, num_classes, concat=True,
#                              heads=self.out_head, dropout=0.6)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
                
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=dropout, training=self.training)
        x = self.conv2(x, edge_index)
        
        return F.log_softmax(x, dim=1)

In [4]:
def train_helper(model, config, ptype):
    device = "cpu"
    if torch.cuda.is_available():
        device = "cuda:0"
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)
    model = model.to(device)
    model = model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=5e-4)
#     optimizer = optim.SGD(net.parameters(), lr=config["lr"], momentum=0.9)

    # Load existing checkpoint through `get_checkpoint()` API.
    if train.get_checkpoint():
        loaded_checkpoint = train.get_checkpoint()
        with loaded_checkpoint.as_directory() as loaded_checkpoint_dir:
            model_state, optimizer_state = torch.load(
                os.path.join(loaded_checkpoint_dir, "checkpoint.pt")
            )
            model.load_state_dict(model_state)
            optimizer.load_state_dict(optimizer_state)
    
    data = load_data(data_dir="/home/k2141255/research/study_design_classifier/data", ptype=ptype)
    data = data.to(device)

    for epoch in range(config["num_epochs"]):  # loop over the dataset multiple times
        optimizer.zero_grad()  # Clear gradients.
        out = model(data)  # Perform a single forward pass.
        loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
        loss.backward()  # Derive gradients.
        optimizer.step()  # Update parameters based on gradients.
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')
        
        
    model = model.eval() 
    with torch.no_grad():
        out = model(data)
            
    pred_val = out[data.val_mask].max(
    labels_val = data.y[data.val_mask]
    val_loss = criterion(pred_val, labels_val)
        
    metric_f = load_metric("f1")
    f1_val = metric_f.compute(predictions=pred_val, references=labels_val)['f1']
    

    with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
        path = os.path.join(temp_checkpoint_dir, "checkpoint.pt")
        torch.save(
            (model.state_dict(), optimizer.state_dict()), path
        )
        checkpoint = Checkpoint.from_directory(temp_checkpoint_dir)
        train.report(
            {"loss": val_loss, "f1": f1_val}, checkpoint=checkpoint,
        )
    del data
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Finished Training")

SyntaxError: '(' was never closed (924941797.py, line 40)

In [22]:
def train_gat_fn(ptype):
    def train_gat(config):
        gat = GAT(config["hid"], config["in_head"], config["out_head"], config["dropout"], num_classes = 2)
        train_helper(gat, config, ptype)
    return train_gat

In [23]:
def test_best_helper(best_trained_model, ptype):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    best_trained_model.to(device)

    checkpoint_path = os.path.join(best_result.checkpoint.to_directory(), "checkpoint.pt")

    model_state, optimizer_state = torch.load(checkpoint_path)
    best_trained_model.load_state_dict(model_state)

    data = load_data(data_dir="/home/k2141255/research/study_design_classifier/data", ptype=ptype)

    best_trained_model = best_trained_model.eval() 
    with torch.no_grad():
        out = best_trained_model(data)
    pred = out.argmax(dim=1)
    
    pred_val = pred[data.val_mask]
    labels_val = data.y[data.val_mask]
    
    metric_acc = load_metric("accuracy")
    metric_p = load_metric("precision")
    metric_r = load_metric("recall")
    metric_f = load_metric("f1")
        
    accuracy_val = metric_acc.compute(predictions=pred_val, references=labels_val)['accuracy']
    precision_val = metric_p.compute(predictions=pred_val, references=labels_val)['precision']
    recall_val = metric_r.compute(predictions=pred_val, references=labels_val)['recall']
    spec_val = metrics.recall_score(labels_val, pred_val, pos_label=0)
    f1_val = metric_f.compute(predictions=pred_val, references=labels_val)['f1']

    pred_test = pred[data.test_mask]
    labels_test = data.y[data.test_mask]
        
    accuracy_test = metric_acc.compute(predictions=pred_test, references=labels_test)['accuracy']
    precision_test = metric_p.compute(predictions=pred_test, references=labels_test)['precision']
    recall_test = metric_r.compute(predictions=pred_test, references=labels_test)['recall']
    spec_test = metrics.recall_score(labels, pred_test, pos_label=0)
    f1_test = metric_f.compute(predictions=pred_test, references=labels_test)['f1']
        
    metrics = {
        "accuracy_val": accuracy_val,
        "precision_val": precision_val,
        "recall_val": recall_val,
        "spec_val": spec_val,
        "f1_val": f1_val,
        "accuracy_test": accuracy_test,
        "precision_test": precision_test,
        "recall_test": recall_test,
        "spec_test": spec_test,
        "f1_test": f1_test
    }
    
    return metrics

In [24]:
def test_best_gat(best_result, ptype):
    best_trained_model = GAT(best_result.config["hid"], best_result.config["in_head"], best_result.config["out_head"], 
                             best_result.config["dropout"], num_classes = 2)
    test_best_helper(best_trained_model, ptype)

In [25]:
config_gat = {
    "hid": tune.choice(list(range(1, 2))),
    "in_head": tune.choice(list(range(1, 2))),
    "out_head": tune.choice(list(range(1, 2))),
    "lr": tune.loguniform(1e-4, 1e-1),
    "num_epochs": tune.choice([20, 50, 100, 200, 300, 400, 500, 600]),
    "dropout": tune.uniform(0, 1)
}

In [29]:
config = config_gat
train_fn = train_gat_fn
test_best = test_best_gat

In [30]:
train_model = train_fn("is_rct")

In [ ]:
            scheduler = ASHAScheduler(
                max_t=3000,
                grace_period=1,
                reduction_factor=2)

            tuner = tune.Tuner(
                tune.with_resources(
                    tune.with_parameters(train_model),
                    resources={"cpu": 3, "gpu": 1}
                ),
                tune_config=tune.TuneConfig(
                    metric="f1",
                    mode="max",
                    scheduler=scheduler,
                    num_samples=5,
                ),
                param_space=config,
            )
            results = tuner.fit()

2024-03-23 12:32:24,565	WARNING tune.py:186 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
